# T3 — Análise e visualização do SIASUS em Ijuí

Este notebook reúne a exploração, preparação, indicadores e gráficos do trabalho. As células estão organizadas para execução de cima para baixo. A conexão com o MySQL e a leitura da view são feitas uma única vez; as análises seguintes trabalham sobre o mesmo DataFrame.

**Fonte e recorte:** produção ambulatorial do SIASUS/DATASUS, mês de movimento julho de 2026, filtrada para estabelecimentos localizados em Ijuí (`pa_ufmun = '431020'`). O uso de um único mês foi combinado com o professor por causa do volume dos dados.

**Antes de começar:** prepare o MySQL nesta ordem: execute `scripts_gerais/script_completo_dbsia.sql`, carregue o CSV PARS em `dbsia.pars`, execute `scripts_gerais/04-gera-dimtempo-mysql.sql` para criar a tabela de granularidade temporal e, por fim, execute a versão atual de `scripts_gerais/view.sql`. A view precisa incluir `PA_PROC_ID` e `PA_TPUPS`. O PARS não está versionado neste repositório.

Os arquivos CSV produzidos pelo notebook são salvos em `analises_geradas/csv/` e os gráficos PNG em `analises_geradas/img/`. Os CSVs de execuções anteriores permanecem nessa pasta até serem substituídos por novas saídas do banco.


## 1. Instalar e importar dependências

Instale os pacotes uma vez no ambiente Python do Jupyter. No VS Code, selecione o mesmo kernel em que instalou as dependências.


In [ ]:
%pip install pandas mysql-connector-python matplotlib seaborn


In [ ]:
from getpass import getpass
from pathlib import Path

import mysql.connector
from mysql.connector import Error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)
sns.set_theme(style="whitegrid")

# A pasta fica na raiz do projeto se o notebook for aberto a partir dela.
RAIZ_PROJETO = Path.cwd()
if not (RAIZ_PROJETO / "scripts_gerais").exists():
    RAIZ_PROJETO = Path.cwd().parent
PASTA_CSV = RAIZ_PROJETO / "analises_geradas" / "csv"
PASTA_IMAGENS = RAIZ_PROJETO / "analises_geradas" / "img"
PASTA_CSV.mkdir(parents=True, exist_ok=True)
PASTA_IMAGENS.mkdir(parents=True, exist_ok=True)
print(f"CSV: {PASTA_CSV}")
print(f"Gráficos: {PASTA_IMAGENS}")

def salvar_csv(dataframe, nome, index=False, index_label=None):
    caminho = PASTA_CSV / nome
    dataframe.to_csv(caminho, index=index, index_label=index_label, encoding="utf-8-sig")
    print(f"CSV salvo: {caminho}")
    return caminho


## 2. Conectar ao MySQL e carregar a view

Informe o usuário e a senha do MySQL local quando solicitado. A consulta usa um cursor bufferizado para consumir o resultado antes de fechar a conexão. Como a view pode ser grande, a leitura pode demorar e ocupa memória.


In [ ]:
usuario_mysql = input("Usuário MySQL: ")
senha_mysql = getpass("Senha MySQL: ")

try:
    conexao = mysql.connector.connect(
        host="localhost",
        user=usuario_mysql,
        password=senha_mysql,
    )
    print("Conexão MySQL estabelecida.")
    with conexao.cursor(buffered=True) as cursor:
        cursor.execute("SELECT * FROM dbsia.pa_ijui")
        colunas = cursor.column_names
        registros = cursor.fetchall()
    conexao.close()
except Error:
    if "conexao" in locals() and conexao.is_connected():
        conexao.close()
    raise

tabela_original = pd.DataFrame(registros, columns=colunas)
print(f"Linhas carregadas: {len(tabela_original):,}")
print(f"Colunas carregadas: {len(tabela_original.columns)}")
display(tabela_original.head())


## 3. Conferir estrutura e requisitos da view

`shape` informa linhas e colunas; `dtypes` mostra os tipos que o conector trouxe. A célula também verifica campos necessários às análises. Se faltar algum, pare e recrie a view com `scripts_gerais/view.sql`.


In [ ]:
print("Dimensão (linhas, colunas):", tabela_original.shape)
print("\nTipos de variáveis:")
display(tabela_original.dtypes.to_frame("Tipo"))

colunas_necessarias = {
    "Ano", "Mes", "Estabelecimento", "PA_PROC_ID", "PA_TPUPS",
    "Procedimento", "Ocupacao", "pa_sexo", "pa_idade", "pa_flidade",
    "pa_cidpri", "Diagnostico", "MunicipioPaciente",
    "pa_qtdpro", "pa_qtdapr", "pa_valpro", "pa_valapr",
}
colunas_faltantes = sorted(colunas_necessarias - set(tabela_original.columns))
if colunas_faltantes:
    raise ValueError(
        "A view está desatualizada ou incompleta. Reexecute scripts_gerais/view.sql. "
        f"Colunas faltantes: {colunas_faltantes}"
    )
print("A view contém as colunas necessárias.")


## 4. Preparar uma cópia para análise

A cópia converte quantidades, valores e idade para números e cria a data de movimento. Valores inválidos viram ausentes para as análises numéricas, mas as linhas não são apagadas.

Não removemos linhas idênticas automaticamente: a view não contém uma chave que permita confirmar se elas são duplicações indevidas do PARS. Códigos de sexo diferentes de `M`/`F` também são preservados e contabilizados.

No leiaute SIASUS, `pa_idade` é a idade em anos; `pa_flidade` indica compatibilidade da idade com a faixa prevista para o procedimento.


In [ ]:
tabela = tabela_original.copy()
colunas_numericas = ["pa_qtdpro", "pa_qtdapr", "pa_valpro", "pa_valapr", "pa_idade"]
for coluna in colunas_numericas:
    tabela[coluna] = pd.to_numeric(tabela[coluna], errors="coerce")

for coluna in ["PA_PROC_ID", "PA_TPUPS", "pa_sexo", "pa_cidpri"]:
    tabela[coluna] = tabela[coluna].astype("string").str.strip()

tabela["DataMovimento"] = pd.to_datetime(
    {
        "year": pd.to_numeric(tabela["Ano"], errors="coerce"),
        "month": pd.to_numeric(tabela["Mes"], errors="coerce"),
        "day": 1,
    },
    errors="coerce",
)
tabela["AnoMes"] = tabela["DataMovimento"].dt.to_period("M").astype("string")

tabela["IdadeValida"] = tabela["pa_idade"].between(0, 120)
bins_idade = [0, 5, 10, 15, 20, 30, 40, 50, 60, 70, 80, 121]
rotulos_idade = ["0-4", "5-9", "10-14", "15-19", "20-29", "30-39",
                 "40-49", "50-59", "60-69", "70-79", "80+"]
tabela["FaixaEtaria"] = pd.cut(
    tabela["pa_idade"].where(tabela["IdadeValida"]),
    bins=bins_idade, labels=rotulos_idade, right=False,
)

print("Linhas preservadas na preparação:", len(tabela))
print("Linhas idênticas na view (não removidas):", int(tabela.duplicated().sum()))


## 5. Qualidade dos dados e exploração inicial

O resumo conta ausências por coluna e códigos demográficos/CID fora dos valores usuais. As contagens são de registros, não de pacientes únicos.


In [ ]:
ausentes = tabela.isna().sum().sort_values(ascending=False).rename("ValoresAusentes")
display(ausentes.to_frame())
salvar_csv(ausentes.to_frame(), "valores_ausentes.csv", index=True, index_label="Coluna")

print("\nFrequência dos códigos de sexo:")
frequencia_sexo = tabela["pa_sexo"].value_counts(dropna=False).rename_axis("CodigoSexo").to_frame("Registros")
display(frequencia_sexo)
salvar_csv(frequencia_sexo, "frequencia_sexo.csv", index=True, index_label="CodigoSexo")

print("\nIdades ausentes ou fora do intervalo 0-120:", int((~tabela["IdadeValida"]).sum()))
print("Códigos de compatibilidade PA_FLIDADE:")
display(tabela["pa_flidade"].value_counts(dropna=False).rename_axis("PA_FLIDADE").to_frame("Registros"))

colunas_descritivas = ["pa_qtdpro", "pa_qtdapr", "pa_valpro", "pa_valapr"]
display(tabela[colunas_descritivas].describe().T)
salvar_csv(tabela[colunas_descritivas].describe().T, "estatisticas_descritivas.csv", index=True, index_label="Variavel")


In [ ]:
nulos_plot = ausentes[ausentes > 0].sort_values()
if not nulos_plot.empty:
    ax = nulos_plot.plot(kind="barh", figsize=(9, 4), color="steelblue")
    ax.set_title("Valores ausentes por atributo")
    ax.set_xlabel("Registros ausentes")
    ax.set_ylabel("Atributo")
    plt.tight_layout()
    plt.savefig(PASTA_IMAGENS / "valores_ausentes.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Nenhum valor ausente encontrado nas colunas da view.")


## 6. Indicadores gerais de produção e recursos financeiros

`pa_qtdpro` e `pa_valpro` são apresentados; `pa_qtdapr` e `pa_valapr` são aprovados. O gasto médio aprovado é calculado como razão entre valor aprovado total e quantidade aprovada total, evitando tirar média simples de médias por linha.


In [ ]:
totais = pd.DataFrame([{
    "Registros": len(tabela),
    "QuantidadeApresentada": tabela["pa_qtdpro"].sum(),
    "QuantidadeAprovada": tabela["pa_qtdapr"].sum(),
    "ValorApresentado": tabela["pa_valpro"].sum(),
    "ValorAprovado": tabela["pa_valapr"].sum(),
}])
totais["DiferencaQuantidade"] = totais["QuantidadeApresentada"] - totais["QuantidadeAprovada"]
totais["DiferencaValor"] = totais["ValorApresentado"] - totais["ValorAprovado"]
totais["GastoMedioAprovadoPorUnidade"] = np.where(
    totais["QuantidadeAprovada"] > 0,
    totais["ValorAprovado"] / totais["QuantidadeAprovada"],
    np.nan,
)
display(totais.T.rename(columns={0: "Valor"}))
salvar_csv(totais, "indicadores_gerais.csv")


In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 5))
totais[["QuantidadeApresentada", "QuantidadeAprovada"]].T.plot(
    kind="bar", ax=eixos[0], color=["steelblue"]
)
eixos[0].set_title("Quantidade apresentada e aprovada")
eixos[0].set_ylabel("Procedimentos")
eixos[0].legend(["Total"])
totais[["ValorApresentado", "ValorAprovado"]].T.plot(
    kind="bar", ax=eixos[1], color=["darkorange"]
)
eixos[1].set_title("Valores apresentados e aprovados")
eixos[1].set_ylabel("R$")
eixos[1].legend(["Total"])
plt.tight_layout()
plt.savefig(PASTA_IMAGENS / "producao_apresentada_aprovada.png", dpi=160, bbox_inches="tight")
plt.show()


## 7. Mês, bimestre e trimestre

O PARS disponível para a análise contém somente julho de 2026, como combinado com o professor. O agrupamento exibe os períodos e deixa claro quantos meses têm dados. Não é possível concluir tendência, picos ou quedas com um único mês.


In [ ]:
mensal = (
    tabela.groupby(["Ano", "Mes", "AnoMes"], dropna=False, as_index=False)
    .agg(
        Registros=("AnoMes", "size"),
        QuantidadeApresentada=("pa_qtdpro", "sum"),
        QuantidadeAprovada=("pa_qtdapr", "sum"),
        ValorApresentado=("pa_valpro", "sum"),
        ValorAprovado=("pa_valapr", "sum"),
    )
    .sort_values(["Ano", "Mes"])
)
bimestral = (tabela.groupby("AnoBimestre", dropna=False, as_index=False)
    .agg(MesesComDados=("AnoMes", "nunique"), Registros=("AnoMes", "size"),
         QuantidadeApresentada=("pa_qtdpro", "sum"),
         QuantidadeAprovada=("pa_qtdapr", "sum"),
         ValorApresentado=("pa_valpro", "sum"),
         ValorAprovado=("pa_valapr", "sum")))
trimestral = (tabela.groupby("AnoTrimestre", dropna=False, as_index=False)
    .agg(MesesComDados=("AnoMes", "nunique"), Registros=("AnoMes", "size"),
         QuantidadeApresentada=("pa_qtdpro", "sum"),
         QuantidadeAprovada=("pa_qtdapr", "sum"),
         ValorApresentado=("pa_valpro", "sum"),
         ValorAprovado=("pa_valapr", "sum")))
print("Produção mensal:")
display(mensal)
print("Bimestre:")
display(bimestral)
print("Trimestre:")
display(trimestral)
salvar_csv(mensal, "producao_mensal.csv")
salvar_csv(bimestral, "producao_bimestral.csv")
salvar_csv(trimestral, "producao_trimestral.csv")
if tabela["AnoMes"].nunique() <= 1:
    print("Há somente um mês disponível: evolução temporal e comparação histórica não se aplicam a esta execução.")


In [ ]:
if len(mensal) > 1:
    ax = mensal.plot(x="AnoMes", y=["QuantidadeApresentada", "QuantidadeAprovada"],
                     marker="o", figsize=(10, 4))
    ax.set_title("Produção ambulatorial por mês")
    ax.set_xlabel("Mês de movimento")
    ax.set_ylabel("Procedimentos")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(PASTA_IMAGENS / "producao_mensal.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Gráfico de tendência não gerado: a base tem um único mês, conforme o recorte acordado.")


## 8. Procedimentos, ocupações e gasto médio por procedimento

O ranking usa quantidade aprovada. A média unitária por procedimento é `valor aprovado / quantidade aprovada` para cada grupo, com `NaN` quando a quantidade aprovada é zero. Essa medida não representa necessariamente o custo por paciente ou por atendimento.


In [ ]:
def resumir_grupo(coluna):
    resumo = (
        tabela.groupby(coluna, dropna=False, as_index=False)
        .agg(Registros=(coluna, "size"),
             QuantidadeApresentada=("pa_qtdpro", "sum"),
             QuantidadeAprovada=("pa_qtdapr", "sum"),
             ValorApresentado=("pa_valpro", "sum"),
             ValorAprovado=("pa_valapr", "sum"))
    )
    resumo["GastoMedioAprovadoPorUnidade"] = (
        resumo["ValorAprovado"] / resumo["QuantidadeAprovada"].where(resumo["QuantidadeAprovada"].ne(0))
    )
    resumo["GastoMedioApresentadoPorUnidade"] = (
        resumo["ValorApresentado"] / resumo["QuantidadeApresentada"].where(resumo["QuantidadeApresentada"].ne(0))
    )
    return resumo.sort_values("QuantidadeAprovada", ascending=False)

procedimentos = resumir_grupo("Procedimento")
ocupacoes = resumir_grupo("Ocupacao")
print("20 procedimentos por quantidade aprovada:")
display(procedimentos.head(20))
print("20 ocupações/especialidades por quantidade aprovada:")
display(ocupacoes.head(20))
salvar_csv(procedimentos, "producao_por_procedimento.csv")
salvar_csv(ocupacoes, "producao_por_ocupacao.csv")
salvar_csv(procedimentos.head(20), "top20_procedimentos.csv")
salvar_csv(ocupacoes.head(20), "top20_ocupacoes.csv")


In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(15, 6))
for eixo, dados_grafico, coluna_rotulo, titulo in [
    (eixos[0], procedimentos.head(10).sort_values("QuantidadeAprovada"), "Procedimento", "10 procedimentos mais aprovados"),
    (eixos[1], ocupacoes.head(10).sort_values("QuantidadeAprovada"), "Ocupacao", "10 ocupações com maior produção aprovada"),
]:
    dados_grafico.plot(x=coluna_rotulo, y="QuantidadeAprovada", kind="barh", ax=eixo, legend=False, color="teal")
    eixo.set_title(titulo)
    eixo.set_xlabel("Quantidade aprovada")
    eixo.set_ylabel("")
plt.tight_layout()
plt.savefig(PASTA_IMAGENS / "top_procedimentos_ocupacoes.png", dpi=160, bbox_inches="tight")
plt.show()


### Como interpretar o gasto médio

Compare a coluna `GastoMedioAprovadoPorUnidade` entre procedimentos apenas com cautela: cada procedimento pode ter unidade de quantidade diferente. A evolução temporal do gasto médio não é apresentada porque o grupo está usando um mês.


## 9. Produção por estabelecimento

O relatório exploratório anterior encontrou `Estabelecimento` ausente em todas as 65.521 linhas. A célula abaixo calcula e exporta o agrupamento sem esconder os ausentes. Se só houver uma linha ausente, **não interprete isso como ranking de estabelecimentos**: a junção com a tabela CNES precisa ser verificada no banco.


In [ ]:
estabelecimentos = (
    tabela.groupby("Estabelecimento", dropna=False, as_index=False)
    .agg(Registros=("Procedimento", "size"),
         QuantidadeApresentada=("pa_qtdpro", "sum"),
         QuantidadeAprovada=("pa_qtdapr", "sum"),
         ValorApresentado=("pa_valpro", "sum"),
         ValorAprovado=("pa_valapr", "sum"))
    .sort_values("QuantidadeAprovada", ascending=False)
)
display(estabelecimentos.head(20))
salvar_csv(estabelecimentos, "producao_por_estabelecimento.csv")
estabelecimentos_identificados = estabelecimentos[estabelecimentos["Estabelecimento"].notna()]
if estabelecimentos_identificados.empty:
    print("ATENÇÃO: não há nomes de estabelecimentos na view; ranking não disponível até corrigir a junção CNES.")
else:
    graf_est = estabelecimentos_identificados.nlargest(10, "QuantidadeAprovada").sort_values("QuantidadeAprovada")
    ax = graf_est.plot(x="Estabelecimento", y="QuantidadeAprovada", kind="barh", legend=False, figsize=(10, 5))
    ax.set_title("Produção aprovada por estabelecimento")
    ax.set_xlabel("Quantidade aprovada")
    plt.tight_layout()
    plt.savefig(PASTA_IMAGENS / "producao_por_estabelecimento.png", dpi=160, bbox_inches="tight")
    plt.show()


## 10. Perfil demográfico por sexo e faixa etária

As categorias de sexo encontradas são preservadas — inclusive o código `0`. As faixas etárias incluem idades entre 0 e 120 anos; registros com idade ausente ou fora dessa faixa permanecem na base, mas não entram no cruzamento etário.


In [ ]:
perfil_sexo = tabela["pa_sexo"].value_counts(dropna=False).rename_axis("PA_SEXO").to_frame("Registros")
perfil_etario_sexo = pd.crosstab(tabela["FaixaEtaria"], tabela["pa_sexo"], dropna=True)
print("Distribuição por sexo:")
display(perfil_sexo)
print("Registros por faixa etária e código de sexo:")
display(perfil_etario_sexo)
salvar_csv(perfil_sexo, "perfil_por_sexo.csv", index=True, index_label="PA_SEXO")
salvar_csv(perfil_etario_sexo, "resumo_perfil_demografico.csv", index=True, index_label="FAIXA_ETARIA")


In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(14, 5))
perfil_sexo["Registros"].plot(kind="bar", ax=eixos[0], color="slateblue")
eixos[0].set_title("Registros por código de sexo")
eixos[0].set_xlabel("Código de sexo")
eixos[0].set_ylabel("Registros")

sns.heatmap(perfil_etario_sexo, annot=True, fmt="d", cmap="Blues", ax=eixos[1])
eixos[1].set_title("Registros por faixa etária e sexo")
eixos[1].set_xlabel("Código de sexo")
eixos[1].set_ylabel("Faixa etária")
plt.tight_layout()
plt.savefig(PASTA_IMAGENS / "perfil_demografico.png", dpi=160, bbox_inches="tight")
plt.show()


## 11. Diagnósticos principais (CID)

A contagem é por registro, não por paciente. São separados códigos e descrições que representam ausência, invalidade ou falta de identificação; eles não aparecem como diagnósticos clínicos no ranking.


In [ ]:
cid = tabela["pa_cidpri"].astype("string").str.strip()
diagnostico = tabela["Diagnostico"].astype("string").str.strip()
diagnostico_normalizado = diagnostico.str.upper()
rotulos_cid_especial = {"CID NAO INFORMADO", "CID NÃO INFORMADO", "CID NÃO IDENTIFICADO", "CID INVALIDO"}
sem_cid_utilizavel = (
    cid.isna() | cid.isin(["", "0000", "9999"])
    | diagnostico_normalizado.isin(rotulos_cid_especial)
)
print(f"Registros sem diagnóstico CID utilizável: {int(sem_cid_utilizavel.sum()):,} de {len(tabela):,}")
top_diagnosticos = (
    diagnostico.loc[~sem_cid_utilizavel & diagnostico.notna()]
    .value_counts().head(10).rename_axis("Diagnostico").to_frame("Registros")
)
display(top_diagnosticos)
salvar_csv(top_diagnosticos, "resumo_top10_diagnosticos.csv", index=True, index_label="Diagnostico")


In [ ]:
if not top_diagnosticos.empty:
    ax = top_diagnosticos["Registros"].sort_values().plot(kind="barh", figsize=(10, 6), color="steelblue")
    ax.set_title("10 diagnósticos mais frequentes com CID utilizável")
    ax.set_xlabel("Registros")
    ax.set_ylabel("Diagnóstico")
    plt.tight_layout()
    plt.savefig(PASTA_IMAGENS / "top10_diagnosticos.png", dpi=160, bbox_inches="tight")
    plt.show()


## 12. Fluxos regionais e município de origem

A view fornece a descrição do município de origem. O valor `INVALIDO` observado no relatório anterior é uma categoria especial da tabela auxiliar, não um município. O resumo abaixo mantém as descrições originais e identifica esse valor para não apresentá-lo como cidade. As contagens são de registros de produção, não de pessoas únicas.


In [ ]:
municipios = tabela["MunicipioPaciente"].fillna("Não informado").astype("string").str.strip()
frequencia_municipios = municipios.value_counts(dropna=False).rename_axis("MunicipioPaciente").to_frame("Registros")
display(frequencia_municipios.head(20))
salvar_csv(frequencia_municipios.head(20), "top20_municipios_origem.csv", index=True, index_label="MunicipioPaciente")

rotulo_invalido = municipios.str.upper().eq("INVALIDO")
municipio_ijui = municipios.str.upper().eq("IJUI")
print(f"Registros com descrição INVALIDO: {int(rotulo_invalido.sum()):,}")
municipio_ausente = municipios.isna() | municipios.eq("") | municipios.eq("Não informado")
municipio_externo_identificado = ~(rotulo_invalido | municipio_ijui | municipio_ausente)
print(f"Registros com município informado como Ijuí: {int(municipio_ijui.sum()):,}")
print(f"Registros de outros municípios identificados: {int(municipio_externo_identificado.sum()):,}")
print(f"Registros sem município informado: {int(municipio_ausente.sum()):,}")
print("Registros INVALIDO permanecem separados e não são contados como município externo.")
print("As contagens são de registros de produção, não de pacientes únicos.")
resumo_origem = pd.DataFrame([{
    "Ijuí": int(municipio_ijui.sum()),
    "OutrosMunicipiosIdentificados": int(municipio_externo_identificado.sum()),
    "MunicipioInvalido": int(rotulo_invalido.sum()),
    "SemMunicipioInformado": int(municipio_ausente.sum()),
}])
salvar_csv(resumo_origem, "resumo_fluxos_regionais.csv")


In [ ]:
municipios_grafico = frequencia_municipios.drop(index=[i for i in frequencia_municipios.index if str(i).upper() == "INVALIDO"], errors="ignore").head(10).sort_values("Registros")
if not municipios_grafico.empty:
    ax = municipios_grafico["Registros"].plot(kind="barh", figsize=(9, 5), color="darkcyan")
    ax.set_title("Municípios de origem mais frequentes (sem categoria INVALIDO)")
    ax.set_xlabel("Registros")
    ax.set_ylabel("Município")
    plt.tight_layout()
    plt.savefig(PASTA_IMAGENS / "top10_municipios_origem.png", dpi=160, bbox_inches="tight")
    plt.show()


## 13. Áreas críticas: oncologia, saúde mental e atenção básica

Os recortes abaixo replicam a regra usada pelos scripts do grupo: oncologia por código de procedimento iniciado em `0304`; saúde mental por CID iniciado em `F` ou tipo de unidade `70`; atenção básica por `PA_TPUPS` em `01` ou `02`. Esses filtros são regras de análise do grupo e precisam ser apresentados junto dos resultados.

Se Atenção Básica resultar zero, verifique a frequência de `PA_TPUPS` antes de concluir que não houve produção.


In [ ]:
codigo_proc = tabela["PA_PROC_ID"].fillna("").astype("string").str.strip().str.zfill(10)
codigo_cid = tabela["pa_cidpri"].fillna("").astype("string").str.strip().str.upper()
tipo_unidade = tabela["PA_TPUPS"].fillna("").astype("string").str.strip().str.zfill(2)

mascaras_areas = {
    "Oncologia": codigo_proc.str.startswith("0304", na=False),
    "Saude mental": codigo_cid.str.startswith("F", na=False) | tipo_unidade.eq("70"),
    "Atencao basica": tipo_unidade.isin(["01", "02"]),
}
linhas_areas = []
for nome, mascara in mascaras_areas.items():
    parte = tabela.loc[mascara]
    linhas_areas.append({
        "Area": nome,
        "Registros": len(parte),
        "qtd_apresentada": parte["pa_qtdpro"].sum(),
        "qtd_aprovada": parte["pa_qtdapr"].sum(),
        "valor_apresentado": parte["pa_valpro"].sum(),
        "valor_aprovado": parte["pa_valapr"].sum(),
    })
resumo_areas = pd.DataFrame(linhas_areas).set_index("Area")
display(resumo_areas)
salvar_csv(resumo_areas, "resumo_areas_criticas.csv", index=True, index_label="Area")
print("\nFrequência de PA_TPUPS para validar o recorte de Atenção Básica:")
display(tipo_unidade.value_counts(dropna=False).rename_axis("PA_TPUPS").to_frame("Registros"))


In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
resumo_areas[["qtd_apresentada", "qtd_aprovada"]].plot(kind="bar", ax=eixos[0])
eixos[0].set_title("Produção apresentada e aprovada por área")
eixos[0].set_ylabel("Procedimentos")
eixos[0].tick_params(axis="x", rotation=15)
resumo_areas[["valor_apresentado", "valor_aprovado"]].plot(kind="bar", ax=eixos[1])
eixos[1].set_title("Valores apresentados e aprovados por área")
eixos[1].set_ylabel("R$")
eixos[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(PASTA_IMAGENS / "areas_criticas.png", dpi=160, bbox_inches="tight")
plt.show()


## 14. Leitura dos resultados e limitações

Use as saídas acima para redigir interpretações com base nos números efetivamente gerados no MySQL. Antes de concluir, responda: quais procedimentos/ocupações concentram produção aprovada? Quais diagnósticos clínicos aparecem mais? Como diferem quantidades e valores apresentados/aprovados? Quais municípios aparecem na origem? Quais áreas críticas se destacam?

Limitações conhecidas do recorte atual:

- Só há um mês de dados (2026-07), por isso não se concluem tendências nem evolução do gasto médio.
- O relatório anterior tinha `Estabelecimento` ausente em todas as linhas. Até corrigir a junção CNES, não há ranking confiável por estabelecimento.
- `INVALIDO` é uma categoria da tabela auxiliar, não cidade; não deve contar como morador de município externo.
- A view não identifica pessoas únicas; métricas são por registro/quantidade produzida, conforme cada tabela.
- Códigos de sexo, idades inválidas, CID ausente e resultados zerados devem ser descritos como qualidade/limitações dos dados, sem remover registros automaticamente.

### Conclusão do grupo

> Preencha após revisar os resultados e discutir com os integrantes: principais achados, possíveis problemas da informação e recomendações para a gestão municipal.
